# Credibility-Weighted Pricing of Autonomous Vehicle Liability — Empirical Walkthrough

This notebook reproduces the empirical results of Section 6 of the paper:

- §6.2 Benchmark reproduction
- §6.3 Posterior estimates under the hierarchical model
- §6.4 Prospective estimation for hypothetical new deployments
- §6.5 Comparison against baselines
- §4.3 Bühlmann–Straub closed-form sanity check

All data are synthetic and generated by `src/data_generator.py` to match the *structure* of the public sources described in §6.1 (NHTSA SGO, FHWA VMT, HLDI, OSM, ACS, FARS). Practitioners with access to real insurer claims data can substitute their own arrays into the same model functions.

## 0. Setup

Run from the repository root. If you haven't generated data yet, the next cell will do so.

In [ ]:
import json, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, '../src')

DATA = Path('../data')
RES = Path('../results')
FIG = Path('../figures')

if not (DATA / 'ads_events.csv').exists():
    from data_generator import generate_all
    summary = generate_all(DATA)
    print(json.dumps(summary, indent=2))

## §6.2 Benchmark reproduction

We approximate the Di Lillo et al. (2024) reduction by comparing the synthetic ADS claim frequency to the matched HDV benchmark in each operating region.

In [ ]:
ads = pd.read_csv(DATA / 'ads_events.csv')
hdv = pd.read_csv(DATA / 'hdv_claims_by_cell.csv')
bench = json.loads((RES / 'benchmark_reproduction.json').read_text())

print(f"Overall ADS frequency:   {bench['overall_ads_freq']:.3f} per million miles")
print(f"Overall HDV frequency:   {bench['overall_hdv_freq']:.3f} per million miles")
print(f"Overall reduction:       {bench['overall_reduction_vs_hdv']*100:.1f}%")
print()
print("Compare to Di Lillo et al. (2024): 88% (PD) and 92% (BI) reduction")
pd.DataFrame(bench['by_city'])

## §4.3 Bühlmann–Straub closed-form sanity check

Before fitting the hierarchical Bayesian model, we compute the classical Bühlmann–Straub credibility weights from city-aggregate totals. Section 4.3 proves the hierarchical Bayesian model nests this classical form as a limiting case; here we verify directly that the classical answer is well-defined and consistent.

**Expected finding:** with ~7 claims spread across 4 cities, the credibility weights Z are essentially zero — the classical formula correctly says we have no signal to deviate from the pooled mean. This is exactly the problem that motivates the hierarchical Bayesian approach with learned ODD similarity.

In [ ]:
bs = pd.read_csv(RES / 'buhlmann_straub_closed_form.csv')
bs

## §6.3 Posterior estimates under the hierarchical model

MCMC diagnostics first. R-hat values close to 1.0 indicate good chain mixing.

In [ ]:
diag = pd.read_csv(RES / 'diagnostics_gp.csv', index_col=0)
print(f"Worst R-hat: {pd.to_numeric(diag['r_hat'], errors='coerce').max():.3f}")
print(f"Worst ESS-bulk: {pd.to_numeric(diag['ess_bulk'], errors='coerce').min():.0f}")
diag.head(10)

In [ ]:
lam = pd.read_csv(RES / 'posterior_lambda_gp.csv')
# Aggregate to (city, version) for display
agg = (lam.groupby(['city', 'version'])
          .agg(lam_mean=('lam_mean', 'mean'),
               lam_q025=('lam_q025', 'mean'),
               lam_q975=('lam_q975', 'mean'),
               exposure=('exposure_million_miles', 'sum'),
               claims=('claims', 'sum'))
          .reset_index())
agg.round(4)

## §6.4 Prospective estimation for hypothetical new deployments

Miami, Boston, and Denver have no ADS exposure in the training set. The GP-prior model produces a posterior predictive for each based on its ODD-embedding similarity to the four deployed cities.

In [ ]:
prosp = pd.read_csv(RES / 'prospective_new_cities.csv')
prosp.round(4)

Note that we report the **posterior median** as the headline point estimate. On the log-scale linear predictor the posterior is approximately Gaussian, but after the exp() link a long right tail develops — most pronounced for Miami, whose maximum similarity to a deployed city (0.92, Los Angeles) is the lowest of the three. The median is the coherent point summary for regulator-facing rate filings; the mean can be inflated by an order of magnitude relative to the median for cells with low similarity. Denver shows the tightest credible interval because of its 0.99 similarity to Austin.

In [ ]:
pros_after = pd.read_csv(RES / 'prospective_after_first_million.csv')
pros_after.round(4)

The first-million-miles posterior update shows the framework behaving as desired: Boston with one observed claim updates from prior median 0.31 to posterior median ~0.70; Miami and Denver with zero observed claims see their priors pulled toward the much lower observed implicit rate.

## §6.5 Comparison against baselines

Leave-one-city-out predictive log-likelihood across four models:
1. **Single pool** — Bühlmann–Straub limit, all cities exchangeable
2. **Hierarchical (indep)** — Section 4 model, independent random effects
3. **GP (Euclidean)** — Section 5.3 with Euclidean kernel on raw features
4. **GP (learned)** — Section 5.3 with the SimCLR-trained embedding kernel

In [ ]:
loco = pd.read_csv(RES / 'loco_comparison.csv')
loco.round(3)

In [ ]:
totals = pd.Series({
    'Single pool': loco['logp_pool'].sum(),
    'Hierarchical (indep)': loco['logp_indep_re'].sum(),
    'GP (Euclidean)': loco['logp_gp_euclidean'].sum(),
    'GP (learned)': loco['logp_gp_learned'].sum(),
}).sort_values(ascending=False)
print(totals.round(3))
print()
print('Note: with only 4 cities and ~7 total claims the differences between\n'
      'models are small relative to sampling noise. The single-pool baseline\n'
      'is competitive precisely because the closed-form BS credibility correctly\n'
      'pulls everything to the grand mean. The intended use case for the GP\n'
      'kernels is prospective pricing (Section 6.4), not retrospective fit on\n'
      'a tiny training set.')

## Figures

All figures from `figures/` are produced by `src/make_figures.py`.

In [ ]:
from IPython.display import Image, display
for fname in sorted(FIG.glob('*.png')):
    print(fname.name)
    display(Image(filename=str(fname)))